# The Python API

**Dr. Panagiotis Sakagiannis, Dr. Alexandros Marantis**

## About this notebook

The previous notebook ran an experiment exactly as it was stored. This one takes that experiment
apart : we load its configuration, change the odor source, look at the larva groups, swap the model
one group uses, and launch the modified version.

That is the whole workflow. Larvaworld configurations are nested dictionaries with typed,
documented fields, so *customizing an experiment* means assigning to a dictionary key - there is no
separate scripting language and no configuration files to write.

**What you will be able to do afterwards**

- Load any stored experiment, model or environment configuration and read it.
- Change a value at any depth of a configuration with plain attribute access.
- Understand what a **larva group** is and how it connects an experiment to a model.
- Launch an experiment built from your own modified configuration.

**Prerequisites** : [Your first simulation](single_simulation.ipynb).

**Cost** : seconds. The one simulation at the end is guarded by a switch.

**Switches in this notebook**

| switch | default | what it turns on |
|---|---|---|
| `RUN_SIM_DEMO` | `False` | the simulation of the customized experiment |
| `RUN_GUI_DEMO` | `False` | the same run with a pygame window |
| `SAVE_MEDIA` | `False` | writing the run to a video file |

## Setup

In [1]:
%matplotlib inline

%load_ext param.ipython

import larvaworld as lw
from larvaworld.lib import reg, sim
from larvaworld.lib.reg.generators import ExpConf

lw.VERBOSE = 1

# Tutorial safety switches
RUN_SIM_DEMO = False
RUN_GUI_DEMO = False
SAVE_MEDIA = False
MEDIA_DIR = "./media"

EXPERIMENT_ID = "chemorbit"

Welcome to the param IPython extension! (https://param.holoviz.org/)
Available magics: %params


Initializing larvaworld registry


Registry configured!


## Section 1 : What a configuration contains

A Larvaworld experiment is a set of configuration variables describing every aspect of a virtual
assay :

- the **environment** - dish geometry and dimensions,
- the **sensory objects** in it - food patches, odor sources, their position, shape and intensity,
- the **larva groups** - how many animals, of which model, placed where,
- the **agent configuration** per group - body physics, behavioral modules and their parameters,
  initial states such as the metabolic one,
- the **recording and analysis** - which parameters are collected and what is computed afterwards.

That is a lot of parameters, which is exactly why the platform ships with a catalog of
preconfigured experiments : you rarely start from nothing, you start from the closest existing one
and change what matters for your question.

`%params` prints the complete schema, with the type, default, bounds and documentation of every
field. It is worth skimming once - after that you will mostly navigate by dot access.

In [2]:
%params ExpConf

## Section 2 : Loading and modifying an experiment

Each preconfigured experiment has a unique ID. `reg.conf.Exp.getID` returns its configuration as a
nested `AttrDict`, which behaves like a dictionary but also supports attribute access - so a deeply
nested value reads like a sentence.

In [3]:
exp_conf = reg.conf.Exp.getID(EXPERIMENT_ID)

print(f"Odor of the source in {EXPERIMENT_ID} :")
print(exp_conf.env_params.food_params.source_units.Source.odor)

Odor of the source in chemorbit :
{'id': 'Odor', 'intensity': 2.0, 'spread': 0.01}


Changing a parameter is an ordinary assignment. Here we sharpen the odor gradient by reducing its
spread : the concentration falls off faster with distance, so the larvae have a steeper signal to
climb.

In [4]:
exp_conf.env_params.food_params.source_units.Source.odor.spread = 0.01

print("Adjusted odor :")
print(exp_conf.env_params.food_params.source_units.Source.odor)

Adjusted odor :
{'id': 'Odor', 'intensity': 2.0, 'spread': 0.01}


## Section 3 : Larva groups

A **larva group** is how an experiment says *put N animals of this kind, there*. Each group has its
own model, its own count, its own spatial distribution and its own color, so that a rendered
simulation immediately shows which agent belongs to which group. Comparing two groups in one dish
is the virtual equivalent of running a control and a treatment side by side - without the lab work.

In [5]:
larva_groups = exp_conf.larva_groups

print("Group IDs :", larva_groups.keylist)
print()
for gID, g in larva_groups.items():
    print(f"{gID:14s} model={g.model!r}  N={g.distribution.N}  color={g.color!r}")

Group IDs : ['navigator']

navigator      model='navigator'  N=3  color='black'


The ID of a group defaults to the ID of the model it uses, which is why the two often look the
same. The `model` field is the link between the experiment and the model catalog : it names a
configuration stored under the `Model` conftype.

In [6]:
group_id = larva_groups.keylist[0]
model_id = larva_groups[group_id].model

print(f"Group {group_id!r} uses model {model_id!r}")

Group 'navigator' uses model 'navigator'


## Section 4 : The model behind a group

Following that link gives the model configuration : the body, the physics and the brain, where the
brain is a set of behavioral modules. Each module can be absent (`None`), or present in one of
several **modes** - alternative implementations of the same function.

In [7]:
m_conf = reg.conf.Model.getID(model_id)

print("Model sections :", m_conf.keylist)
print()
print("Brain modules :")
for k, v in m_conf.brain.items():
    if isinstance(v, dict) and "mode" in v:
        print(f"  {k:16s} mode={v.mode!r}")
    elif v is None:
        print(f"  {k:16s} -")

Model sections : ['brain', 'body', 'physics', 'energetics', 'sensorimotor', 'Box2D']

Brain modules :
  crawler          mode='realistic'
  interference     mode='phasic'
  turner           mode='neural'
  intermitter      mode='default'
  feeder           -
  olfactor         mode='default'
  toucher          -
  windsensor       -
  thermosensor     -
  memory           -


Swapping the model a group uses is, again, one assignment. `Levy_navigator` navigates towards odor
just like `navigator` does, but its exploratory bouts follow a Levy distribution instead of the
coupled oscillator - a different search strategy on top of the same sensory machinery.

In [8]:
mIDs = reg.conf.Model.confIDs
print(f"{len(mIDs)} stored model configurations, e.g.")
print([m for m in mIDs if "navigator" in m][:10])

612 stored model configurations, e.g.
['Levy_navigator', 'Levy_navigator0', 'Levy_navigator0_MB', 'Levy_navigator0_RL', 'Levy_navigator0_brute', 'Levy_navigator0_brute_MB', 'Levy_navigator0_brute_RL', 'Levy_navigator_MB', 'Levy_navigator_RL', 'Levy_navigator_brute']


In [9]:
exp_conf.larva_groups[group_id].model = "Levy_navigator"

print(f"Group {group_id!r} now uses {exp_conf.larva_groups[group_id].model!r}")

Group 'navigator' now uses 'Levy_navigator'


## Section 5 : Running the customized experiment

`ExpRun` accepts either a stored experiment ID with overrides, or a full `parameters` dictionary.
Passing `modelIDs` is a shortcut for the common case of *run this experiment with these models, one
group each* - which is what we do here to put the two navigators in the same dish.

In [10]:
run_id = "my-custom-run"

screen_kws = {}
if RUN_GUI_DEMO or SAVE_MEDIA:
    screen_kws = {
        "vis_mode": "video",
        "show_display": RUN_GUI_DEMO,
        "save_video": SAVE_MEDIA,
        "fps": 20,
        # saved as {media_dir}/larva-sim-{run_id}.mp4
        "video_file": f"larva-sim-{run_id}",
        "media_dir": MEDIA_DIR,
    }

erun = sim.ExpRun(
    experiment=EXPERIMENT_ID,
    modelIDs=["navigator", "Levy_navigator"],
    screen_kws=screen_kws,
    N=2,
    duration=0.5,  # minutes
)

print("Groups in this run :", erun.p.larva_groups.keylist)

Groups in this run : ['navigator', 'Levy_navigator']


In [11]:
if RUN_SIM_DEMO:
    erun.simulate()
    print(f"Run {run_id} completed, {len(erun.datasets)} dataset(s) produced")

## Where to go next

- [The configuration registry](../4_models_and_environments/configuration_registry.ipynb) - store
  your modified configuration under its own ID so you can reuse it.
- [Building an environment](../4_models_and_environments/environment_configuration.ipynb) - the
  arena side of the configuration, in detail.
- [Custom brain modules](../6_extending_larvaworld/custom_brain_modules.ipynb) - when changing
  parameters is not enough and you need a new module.
- Reference : [Larva agent architecture](../../agents_environments/larva_agent_architecture.md).